# 07: Sentinel Service Verification & HITL Demo

This notebook demonstrates the **VeNRA Sentinel Service** in action. It validates an agent's response by:
1. Splitting the response into sentences.
2. Performing a "White-Box" groundedness check via the fine-tuned SLM Judge.
3. Aggregating scores for an overall reliability metric.

We will perform a live run against the Hugging Face hosted Judge and conduct a manual verification of its labels.

In [6]:
import requests
import json
import pandas as pd
from IPython.display import display, HTML

SENTINEL_URL = "http://localhost:8000/verify"
HF_JUDGE_URL = "https://pagand-venra-haldet.hf.space/verify"

## 0. Connectivity & Raw Response Pre-Check
Before testing the local Sentinel Service, we verify the Hugging Face Space and inspect the **RAW JSON** output to ensure key mapping is correct.

In [7]:
def test_hf_direct():
    payload = {
        "sentence": "Revenue for 2023 was $383 billion.", 
        "context": "Total net sales in 2023 were $383,285 million, compared to $394,328 million in 2022.", 
        "trace": "# Extracted revenue: 383e9"
    }
    print(f"Checking Hugging Face Judge at: {HF_JUDGE_URL}...")
    try:
        response = requests.post(HF_JUDGE_URL, json=payload, timeout=60)
        if response.status_code == 200:
            res_json = response.json()
            print("✅ Success! Hugging Face Judge is Online.")
            print("--- RAW JSON RESPONSE ---")
            print(json.dumps(res_json, indent=2))
            print("-------------------------")
            return True
        else:
            print(f"❌ Failed! Status: {response.status_code}")
            print(response.text)
            return False
    except Exception as e:
        print(f"❌ Connection Error: {e}")
        return False

hf_online = test_hf_direct()

Checking Hugging Face Judge at: https://pagand-venra-haldet.hf.space/verify...
✅ Success! Hugging Face Judge is Online.
--- RAW JSON RESPONSE ---
{
  "prediction": "UNFOUNDED",
  "probabilities": {
    "supported": 0.208163,
    "unfounded": 0.791807,
    "general": 3e-05
  }
}
-------------------------
